# ReEntryAI Run v2 -- Orion CM Entry with Interval Supervisor and RCS

End-to-end simulation notebook. Drives the corrected 3DOF translational dynamics
with the Orion CM trim aero database, a heat-aware predictor-corrector guidance
law, a physical bank actuator, a PD roll torque controller, and a 12-thruster RCS.
Interval propagation runs alongside the nominal trajectory to provide uncertainty
tubes for every state.

**All outputs are written to `revision_v1/` inside `StochasticEntrySim/`.**

## Sections
0. Setup (imports, helpers, output folder)
1. Configuration (vehicle, guidance, control stack, RCS, supervisor)
2. Initial state
3. Main simulation loop
4. Save raw data
5. Trajectory diagnostics (ground track, 3D descent tube)
6. State time histories with interval bands (altitude, speed, gamma, chi)
7. Interval supervisor diagnostics (widths, density, dynamic pressure)
8. Heating envelope (qdot, Q, heat shield map)
9. Guidance candidate analysis
10. RCS firing analysis
11. Final summary

## 0. Setup

In [ ]:
# Imports
import json
import math
import os
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)

import constants
import AtmosphereModel
import control
import ReactionControl

from interval_math import Interval, promote
from math_3d import (
    IntervalSupervisorConfig,
    annotate_nominal_state_with_interval_supervisor,
    f_interval,
    nominal_state_to_interval_box,
    nominal_aero_forces_from_state,
    nominal_eom_step,
    nominal_heating_envelope_from_state,
)
from control import (
    ReentryState,
    GuidanceScheduler,
    SimpleBankGuidance,
    BasicObservationProvider,
    BankActuatorLimits,
    BankActuator,
    RollTorqueController,
    CapsuleControlConfig,
    CapsuleControlStack,
    wrap_to_pi,
)
from ReactionControl import build_orion_cm_rcs_12
from point_math_3d import make_initial_capsule_attitude, step_closed_loop_milestone1, aero_forces

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
import plotting
import cpas
import mission_config

# Mission config -- single JSON file controls all routinely-varied
# parameters. Change CONFIG_PATH below (or the SIM_CONFIG env var) to
# switch runs.
CONFIG_PATH = 'configs/default.json'
MISSION_CFG = mission_config.load_config(CONFIG_PATH)
print(mission_config.summarize(MISSION_CFG))


In [ ]:
# Output folder comes from the mission config so each config writes
# to its own folder (typically revision_v1/ or runs/<run_id>/).
from pathlib import Path
OUTPUT_DIR = Path(str(MISSION_CFG.output_dir))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir :', OUTPUT_DIR)
print('Figure dir :', FIG_DIR)

def save_fig(name):
    """Save the current figure into the figures folder under the run's output_dir."""
    path = FIG_DIR / f'{name}.png'
    plt.savefig(path, bbox_inches='tight')
    try:
        rel = path.relative_to(OUTPUT_DIR.parent)
    except ValueError:
        rel = path
    print(f'  saved {rel}')


In [ ]:
# Helper functions
STATE_NAMES = ['r_m', 'phi_rad', 'lam_rad', 'V_mps', 'gamma_rad', 'chi_rad']

def deg(x_rad):
    return math.degrees(x_rad)

def alt_from_r(r_m):
    return r_m - constants.RADIUS_EARTH

def wrap_angle_rad(x_rad):
    return wrap_to_pi(float(x_rad))

def atmosphere_from_state_vector(x):
    r_m, _, _, V_mps, _, _ = x
    alt_m = max(0.0, alt_from_r(r_m))
    atm = AtmosphereModel.US_Standard_ATM(alt_m)
    rho = atm['rho_kgm3'] if atm['rho_kgm3'] is not None else 0.0
    T_K = atm['T_K'] if atm['T_K'] is not None else np.nan
    p_Pa = atm['p_Pa'] if atm['p_Pa'] is not None else np.nan
    q_pa = 0.5 * rho * V_mps * V_mps
    return {'alt_m': alt_m, 'rho_kgm3': rho, 'T_K': T_K, 'p_Pa': p_Pa, 'q_pa': q_pa}

## 1. Configuration

In [ ]:
# Vehicle parameters -- use the Orion CM trim aero database we added in math_3d.py
params = {
    'mass_kg': float(constants.CAPSULE_MASS_KG),
    'ref_area_m2': float(constants.CAPSULE_REFERENCE_AREA_M2),
    'aero_model': 'orion_cm_trim',
    'CD_const': 1.15,   # fallback only
    'CL_const': 0.28,
}

dt_s = float(constants.ENTRY_DT_S)
t_final_s = 1000.0
num_steps = int(t_final_s / dt_s)
Izz_kgm2 = float(constants.CAPSULE_IZZ_KGM2)

# Landing target
target_phi_rad = math.radians(15.00)
target_lam_rad = math.radians(15.00)

trajectory_id = 'revision_v1_traj_000'
print('dt_s =', dt_s, ' num_steps =', num_steps, ' t_final_s =', t_final_s)
mission_config.apply_aero_to_params(MISSION_CFG, params)


In [ ]:
# Guidance + control stack
control_cfg = CapsuleControlConfig(guidance_period_s=float(constants.GUIDANCE_PERIOD_S))
guidance_scheduler = GuidanceScheduler(period_s=float(constants.GUIDANCE_PERIOD_S))

guidance_law = SimpleBankGuidance(
    target_phi_rad=target_phi_rad,
    target_lam_rad=target_lam_rad,
    params=dict(params),
    max_bank_deg=70.0,
    min_bank_deg=40.0,
    velocity_enable_mps=40_000.0,
    predictor_horizon_steps=int(constants.PREDICTOR_CORRECTOR_HORIZON_STEPS),
    prediction_dt_s=float(constants.ENTRY_DT_S),
    candidate_bank_deg=[float(v) for v in constants.PREDICTOR_CANDIDATE_BANK_DEG],
    weight_range=float(constants.PREDICTOR_COST_WEIGHT_RANGE),
    weight_heading=float(constants.PREDICTOR_COST_WEIGHT_HEADING),
    weight_cross_track=float(constants.PREDICTOR_COST_WEIGHT_CROSS_TRACK),
    weight_heat=float(constants.PREDICTOR_COST_WEIGHT_HEAT),
    heat_rate_limit=float(constants.HEAT_RATE_LIMIT_DEFAULT),
    heat_load_limit=float(constants.HEAT_LOAD_LIMIT_DEFAULT),
    infeasible_penalty=float(constants.PREDICTOR_HEAT_INFEASIBLE_PENALTY),
    invalid_interval_penalty=float(constants.PREDICTOR_INVALID_INTERVAL_PENALTY),
    include_zero_bank_candidate=True,
)

observation_provider = BasicObservationProvider(
    target_phi_rad=target_phi_rad,
    target_lam_rad=target_lam_rad,
    params=params,
)

bank_actuator = BankActuator(BankActuatorLimits(
    sigma_rate_max_rps=math.radians(20.0),
    sigma_accel_max_rps2=math.radians(40.0),
))

roll_controller = RollTorqueController(
    kp_Nm_per_rad=float(constants.ROLL_KP_NM_PER_RAD),
    kd_Nm_per_rad_s=float(constants.ROLL_KD_NM_PER_RAD_S),
    max_torque_Nm=float(constants.ROLL_CMD_MAX_TORQUE_NM),
    sigma_deadband_rad=float(constants.ROLL_SIGMA_DEADBAND_RAD),
    rate_deadband_rad_s=float(constants.ROLL_RATE_DEADBAND_RAD_S),
)

capsule_control = CapsuleControlStack(
    cfg=control_cfg,
    scheduler=guidance_scheduler,
    guidance=guidance_law,
    obs_provider=observation_provider,
    bank_actuator=bank_actuator,
    roll_controller=roll_controller,
)

rcs_system = build_orion_cm_rcs_12()
capsule_control.reset()
rcs_system.reset()
print('Control stack ready. Roll thrusters +z:', rcs_system.roll_pos_names, '  -z:', rcs_system.roll_neg_names)

In [ ]:
# Interval supervisor -- defines the uncertainty tube and recenter/split policies
supervisor_cfg = IntervalSupervisorConfig(
    r_half_width_m=10.0,
    phi_half_width_rad=math.radians(0.01),
    lam_half_width_rad=math.radians(0.01),
    V_half_width_mps=20.0,
    gamma_half_width_rad=math.radians(0.15),
    chi_half_width_rad=math.radians(0.20),
    min_altitude_m=0.0,
    max_altitude_m=130_000.0,
    max_speed_mps=80_000.0,
    max_dynamic_pressure_pa=5.0e7,
    include_heating=True,
    heat_rate_limit=float(constants.HEAT_RATE_LIMIT_DEFAULT),
    heat_load_limit=float(constants.HEAT_LOAD_LIMIT_DEFAULT),
    interval_recenter_enabled=bool(constants.INTERVAL_RECENTER_ENABLED),
    interval_recenter_use_cadence=False,
    interval_recenter_cadence_s=float(constants.INTERVAL_RECENTER_CADENCE_S),
    interval_recenter_width_thresholds=dict(constants.INTERVAL_RECENTER_WIDTH_THRESHOLDS),
    interval_box_split_enabled=bool(constants.INTERVAL_BOX_SPLIT_ENABLED),
    interval_box_split_max_depth=int(constants.INTERVAL_BOX_SPLIT_MAX_DEPTH),
    interval_box_split_width_thresholds=dict(constants.INTERVAL_BOX_SPLIT_WIDTH_THRESHOLDS),
    interval_denominator_safety_V_mps=float(constants.INTERVAL_DENOMINATOR_SAFETY_V_MPS),
    interval_denominator_safety_cos_gamma=float(constants.INTERVAL_DENOMINATOR_SAFETY_COS_GAMMA),
    interval_denominator_safety_cos_phi=float(constants.INTERVAL_DENOMINATOR_SAFETY_COS_PHI),
)
print('Interval supervisor ready. Recenter enabled =', supervisor_cfg.interval_recenter_enabled,
      ' Splitting enabled =', supervisor_cfg.interval_box_split_enabled)

## 2. Initial state

In [ ]:
# Entry interface conditions and landing target -- all from the mission config.
x_nominal = mission_config.initial_state_vector(MISSION_CFG, float(constants.RADIUS_EARTH))
_targets = mission_config.mission_targets(MISSION_CFG)
target_phi_rad = float(_targets['target_phi_rad'])
target_lam_rad = float(_targets['target_lam_rad'])
COS_GAMMA_TERMINATION_GATE = float(_targets['cos_gamma_termination_gate'])
trajectory_id = MISSION_CFG.run_id

# Build the closed-loop control state from the initial vector
sigma_actual_0_rad = 0.0
roll_rate_0_rad_s  = 0.0
state_ctrl = control.ReentryState(
    r_m=float(x_nominal[0]),
    phi_rad=float(x_nominal[1]),
    lam_rad=float(x_nominal[2]),
    V_mps=float(x_nominal[3]),
    gamma_rad=float(x_nominal[4]),
    chi_rad=float(x_nominal[5]),
    sigma_actual_rad=float(sigma_actual_0_rad),
    roll_rate_rad_s=float(roll_rate_0_rad_s),
    sigma_cmd_rad=0.0,
    sigma_target_rad=0.0,
)

print(f'Initial: alt = {(x_nominal[0]-constants.RADIUS_EARTH)/1000.0:.1f} km, V = {x_nominal[3]:.0f} m/s, gamma = {math.degrees(x_nominal[4]):.2f} deg')
print(f'Target : phi = {math.degrees(target_phi_rad):.2f} deg, lam = {math.degrees(target_lam_rad):.2f} deg')


## 3. Main simulation loop

Each step: control stack -> RCS -> roll attitude -> translational EOM -> interval annotation -> log.

In [ ]:
# Controller bridge -- same shape used in the original notebook
def control_step_fn_main(t_s, x_state, sigma_actual_rad, roll_rate_rad_s):
    r_m, phi_rad, lam_rad, V_mps, gamma_rad, chi_rad = [float(v) for v in x_state]
    state_now = ReentryState(
        r_m=r_m, phi_rad=phi_rad, lam_rad=lam_rad, V_mps=V_mps,
        gamma_rad=gamma_rad, chi_rad=chi_rad,
        sigma_actual_rad=float(sigma_actual_rad),
        roll_rate_rad_s=float(roll_rate_rad_s),
    )
    Dr, Dt, Dp, Lr, Lt, Lp, _, _ = aero_forces(
        r=r_m, V=V_mps, gamma=gamma_rad, chi=chi_rad,
        params=params, sigma_actual_rad=float(sigma_actual_rad),
    )
    drag = math.sqrt(Dr*Dr + Dt*Dt + Dp*Dp)
    lift = math.sqrt(Lr*Lr + Lt*Lt + Lp*Lp)
    return capsule_control.step(
        t_s=float(t_s), dt_s=float(dt_s), state=state_now,
        lift_N=lift, drag_N=drag, mass_kg=float(params['mass_kg']),
    )

In [ ]:
import random as _rng_mod
# Main loop -- preserves the interval supervisor logic from the original notebook.
x_nom = list(x_nominal)
x_interval = None
interval_heat_shield = None
nominal_heat_shield = None
interval_active = True
interval_failed = False
interval_failure_time_s = np.nan
interval_failure_reason = ''
interval_failure_kind = ''
last_valid_interval_state = None
last_valid_interval_time_s = np.nan
nominal_heat_violation = False
sim_terminated_by_nominal_heat = False
termination_reason = ''
att_nom = make_initial_capsule_attitude(state_ctrl.sigma_actual_rad)

rows = []
failed_heat_rows = []
thruster_fire_events = []  # one row per (step, thruster_name) fire

# --- CPAS (parachute) integration ---
_cpas_config = mission_config.build_cpas_config(MISSION_CFG)
cpas_inst = cpas.CPAS(config=_cpas_config, rng=_rng_mod.Random(int(MISSION_CFG.seed)))
cpas_events_log = []   # one row per phase transition
params['cpas_drag_cdA_extra_m2'] = 0.0
params['cpas_lift_scale'] = 1.0

for k in range(num_steps):
    t_s = k * dt_s
    sigma_actual_rad = float(att_nom.sigma_rel_rad)
    roll_rate_rad_s = float(att_nom.omega_b_rad_s[2])

    # --- CPAS state machine: drogue -> pilot -> main triggers ---
    alt_now_m = alt_from_r(x_nom[0])
    T_now = atmosphere_from_state_vector(x_nom).get('T_K', float('nan'))
    mach_now = float(x_nom[3]) / math.sqrt(1.4 * 287.0 * float(T_now)) if T_now and T_now == T_now and T_now > 0 else None
    cpas_out = cpas_inst.step(t_s=t_s, alt_m=alt_now_m, V_mps=float(x_nom[3]), mach=mach_now)
    params['cpas_drag_cdA_extra_m2'] = float(cpas_out.drag_area_cdA_m2)
    params['cpas_lift_scale'] = float(cpas_out.lift_scale)
    for ev in cpas_out.events:
        cpas_events_log.append({'t_s': t_s, 'step': k, 'alt_m': alt_now_m, 'V_mps': float(x_nom[3]),
                                'mach': mach_now if mach_now is not None else float('nan'),
                                'phase': cpas_out.phase, 'event': ev})
        print(f'  CPAS {ev} at t={t_s:.1f}s  alt={alt_now_m:.1f}m  V={float(x_nom[3]):.1f}m/s')

    # cos_gamma safety gate only applies pre-deployment. Once chutes are out
    # the bank/heading channels are dead and a vertical fall is expected.
    if cpas_out.phase == 'stowed':
        if abs(math.cos(x_nom[4])) < COS_GAMMA_TERMINATION_GATE:
            termination_reason = 'nominal_cos_gamma_too_small'
            print(f'Stopping at t = {t_s:.2f} s: cos(gamma) too small')
            break
    if x_nom[3] <= 0.1:
        termination_reason = 'nominal_speed_too_low'
        print(f'Stopping at t = {t_s:.2f} s: speed too low')
        break
    if alt_now_m <= 0.0:
        termination_reason = 'nominal_ground_reached'
        print(f'Stopping at t = {t_s:.2f} s: ground reached')
        break

    guidance_law.set_prediction_context(
        x_interval_old=x_interval if interval_active else None,
        heat_shield=interval_heat_shield if interval_active else None,
        supervisor_cfg=supervisor_cfg,
        trajectory_id=trajectory_id,
        guidance_cycle_index=k,
        interval_active=bool(interval_active),
    )

    step_result = step_closed_loop_milestone1(
        t_s=float(t_s), x_trans=list(x_nom), att=att_nom,
        params=params, control_step_fn=control_step_fn_main,
        rcs=rcs_system, dt_s=float(dt_s), Izz_kgm2=float(Izz_kgm2),
    )
    ctrl_out = step_result.control_out
    roll_step = step_result.roll_step
    x_nom_next = list(step_result.x_new)
    dx_nom = list(step_result.dx_trans)
    sigma_actual_next_rad = float(step_result.sigma_actual_after_rad)
    roll_rate_next_rad_s = float(step_result.att_new.omega_b_rad_s[2])
    roll_accel = (roll_rate_next_rad_s - roll_rate_rad_s) / float(dt_s)

    g_debug = dict(ctrl_out.guidance_debug)
    candidate_dicts = g_debug.get('candidate_dicts', [])
    selected_idx = int(g_debug.get('selected_candidate_index', -1))

    nominal_aero = nominal_aero_forces_from_state(x=x_nom, sigma_rad=sigma_actual_next_rad, params=params)
    gravity = float(constants.gravity(float(x_nom[0])))

    heat_info = nominal_heating_envelope_from_state(x_nominal=list(x_nom), dt_s=float(dt_s), heat_shield=nominal_heat_shield)
    nominal_heat_shield = heat_info['heat_shield']
    nominal_heat_qdot = heat_info['qdot_max']
    nominal_heat_Q = heat_info['Q_max']
    nominal_heat_violation = False
    nominal_heat_violation_reason = ''
    if nominal_heat_qdot.hi > float(supervisor_cfg.heat_rate_limit):
        nominal_heat_violation = True; nominal_heat_violation_reason = 'nominal_heat_rate_limit'
    if nominal_heat_Q.hi > float(supervisor_cfg.heat_load_limit):
        nominal_heat_violation = True
        nominal_heat_violation_reason += (',' if nominal_heat_violation_reason else '') + 'nominal_heat_load_limit'

    annotation = None
    interval_step_recentered = False; interval_step_recenter_reason = ''
    interval_step_split = False; interval_step_split_reason = ''; interval_step_split_depth = 0
    if interval_active:
        annotation = annotate_nominal_state_with_interval_supervisor(
            x_nominal_old=x_nom, params=params,
            sigma_actual_after_rad=sigma_actual_next_rad,
            x_interval_old=x_interval, supervisor_cfg=supervisor_cfg,
            dt_s=dt_s, heat_shield=interval_heat_shield, t_s=t_s,
        )
        interval_step_recentered = bool(annotation.recentered_this_step)
        interval_step_recenter_reason = str(getattr(annotation, 'recenter_reason', ''))
        interval_step_split = bool(annotation.split_this_step)
        interval_step_split_reason = str(getattr(annotation, 'split_reason', ''))
        interval_step_split_depth = int(annotation.split_depth_used)
        if annotation.last_valid_interval_state is not None:
            last_valid_interval_state = [Interval(iv.lo, iv.hi) for iv in annotation.last_valid_interval_state]
            last_valid_interval_time_s = float(t_s)
        if annotation.interval_valid and not annotation.interval_heat_violation:
            x_interval = [Interval(iv.lo, iv.hi) for iv in annotation.x_interval_new]
            interval_heat_shield = getattr(annotation, 'heat_shield', interval_heat_shield)
        else:
            interval_failed = True; interval_active = False
            interval_failure_time_s = float(t_s + dt_s)
            interval_failure_reason = str(annotation.interval_failure_reason)
            interval_failure_kind = str(annotation.interval_failure_kind)
            interval_heat_shield = None; x_interval = None

    atm = atmosphere_from_state_vector(x_nom)
    active_names = roll_step.fire_cmd.active_names()
    obs = ctrl_out.obs

    # Record one row per thruster that fired this step (for raster plot)
    for tn in active_names:
        thruster_fire_events.append({'step': k, 't_s': t_s, 'thruster': tn})

    row = {
        'step': k, 't_s': t_s,
        'guidance_updated': int(bool(ctrl_out.guidance_updated)),
        'trajectory_id': trajectory_id,
        'guidance_cycle_index': int(g_debug.get('guidance_cycle_index', k)),
        # bank / roll
        'sigma_cmd_rad': float(ctrl_out.sigma_cmd_rad),
        'sigma_target_rad': float(ctrl_out.sigma_target_rad),
        'sigma_actual_rad': sigma_actual_next_rad,
        'roll_rate_rad_s': roll_rate_next_rad_s,
        'roll_accel_rad_s2': roll_accel,
        # RCS
        'tau_roll_cmd_Nm': float(ctrl_out.tau_roll_cmd_Nm),
        'tau_roll_capacity_Nm': float(roll_step.tau_roll_capacity_Nm),
        'tau_roll_realized_Nm': float(roll_step.wrench.torque_b_Nm[2]),
        'requested_duty': float(roll_step.requested_duty),
        'fired_this_step': int(bool(roll_step.fired_this_step)),
        'num_internal_steps': int(roll_step.num_internal_steps),
        'num_fired_internal_steps': int(roll_step.num_fired_internal_steps),
        'roll_pos_is_on': int(bool(roll_step.roll_pos_is_on)),
        'roll_neg_is_on': int(bool(roll_step.roll_neg_is_on)),
        'roll_pos_backlog_s': float(roll_step.roll_pos_backlog_s),
        'roll_neg_backlog_s': float(roll_step.roll_neg_backlog_s),
        'active_thrusters': ','.join(active_names),
        'num_active_thrusters': len(active_names),
        'force_x_from_rcs_N': float(roll_step.wrench.force_b_N[0]),
        'force_y_from_rcs_N': float(roll_step.wrench.force_b_N[1]),
        'force_z_from_rcs_N': float(roll_step.wrench.force_b_N[2]),
        # CPAS (parachute) state and contribution
        'cpas_phase': str(cpas_out.phase),
        'cpas_drag_cdA_m2': float(cpas_out.drag_area_cdA_m2),
        'cpas_lift_scale': float(cpas_out.lift_scale),
        'cpas_open_fraction': float(cpas_out.open_fraction),
        'cpas_force_vertical': int(bool(cpas_out.force_vertical)),
        'cpas_events': ','.join(cpas_out.events) if cpas_out.events else '',
        # translational state
        'r_m': float(x_nom[0]), 'phi_rad': float(x_nom[1]), 'lam_rad': float(x_nom[2]),
        'V_mps': float(x_nom[3]), 'gamma_rad': float(x_nom[4]), 'chi_rad': float(x_nom[5]),
        'alt_m': float(atm['alt_m']),
        'rho_kgm3': float(atm['rho_kgm3']),
        'q_pa': float(atm['q_pa']),
        'T_K': float(atm['T_K']) if not pd.isna(atm['T_K']) else np.nan,
        'p_Pa': float(atm['p_Pa']) if not pd.isna(atm['p_Pa']) else np.nan,
        # aero
        'drag_mag_N': float(nominal_aero['drag_mag_N']),
        'lift_mag_N': float(nominal_aero['lift_mag_N']),
        'CD': float(nominal_aero['CD']),
        'CL': float(nominal_aero['CL']),
        'LD': float(nominal_aero['LD']),
        'gravity_mps2': gravity,
        # geometry
        'range_to_go_m': float(obs.get('range_to_go_m', np.nan)),
        'heading_error_rad': float(obs.get('heading_error_rad', np.nan)),
        'cross_track_error_m': float(obs.get('cross_track_error_m', np.nan)),
        'along_track_error_m': float(obs.get('along_track_error_m', np.nan)),
        # interval status
        'interval_active': int(interval_active),
        'interval_failed': int(interval_failed),
        'interval_failure_time_s': float(interval_failure_time_s) if np.isfinite(interval_failure_time_s) else np.nan,
        'interval_failure_reason': str(interval_failure_reason),
        'interval_failure_kind': str(interval_failure_kind),
        'interval_step_recentered': int(interval_step_recentered),
        'interval_step_recenter_reason': interval_step_recenter_reason,
        'interval_step_split': int(interval_step_split),
        'interval_step_split_reason': interval_step_split_reason,
        'interval_step_split_depth': interval_step_split_depth,
        # nominal heat
        'nominal_heat_qdot_max_hi': float(nominal_heat_qdot.hi),
        'nominal_heat_Q_max_hi': float(nominal_heat_Q.hi),
        'nominal_heat_violation': int(nominal_heat_violation),
        # guidance
        'guidance_selected_candidate_index': int(selected_idx),
        'guidance_chosen_sigma_cmd_deg': float(g_debug.get('chosen_sigma_cmd_deg', np.nan)),
        'guidance_chosen_sigma_mag_deg': float(g_debug.get('chosen_sigma_mag_deg', np.nan)),
        'guidance_selected_total_cost': float(g_debug.get('selected_total_cost', np.nan)),
        'guidance_selected_geometry_cost': float(g_debug.get('selected_geometry_cost', np.nan)),
        'guidance_selected_heat_penalty': float(g_debug.get('selected_heat_penalty', np.nan)),
        'guidance_any_feasible': int(bool(g_debug.get('any_feasible_candidate', True))),
        'guidance_candidate_sigma_deg_json': json.dumps([float(c.get('sigma_cmd_deg', np.nan)) for c in candidate_dicts]),
        'guidance_candidate_cost_json': json.dumps([float(c.get('total_cost', np.nan)) for c in candidate_dicts]),
        'guidance_candidate_heat_flag_json': json.dumps([int(bool(c.get('fully_feasible', False))) for c in candidate_dicts]),
    }

    # Interval bands (NaN when interval has failed)
    if annotation is not None:
        row['interval_alt_lo'] = float(annotation.altitude_interval.lo)
        row['interval_alt_hi'] = float(annotation.altitude_interval.hi)
        row['interval_rho_lo'] = float(annotation.rho_interval.lo)
        row['interval_rho_hi'] = float(annotation.rho_interval.hi)
        row['interval_q_lo'] = float(annotation.q_interval.lo)
        row['interval_q_hi'] = float(annotation.q_interval.hi)
        state_base_names = ['r', 'phi', 'lam', 'V', 'gamma', 'chi']
        for i, base in enumerate(state_base_names):
            row[f'{base}_lo'] = float(annotation.x_interval_new[i].lo)
            row[f'{base}_hi'] = float(annotation.x_interval_new[i].hi)
            row[f'{base}_width'] = float(annotation.x_interval_new[i].width())
        row['width_r_m'] = float(annotation.state_widths_new['r'])
        row['width_V_mps'] = float(annotation.state_widths_new['V'])
        row['width_gamma_rad'] = float(annotation.state_widths_new['gamma'])
        row['width_chi_rad'] = float(annotation.state_widths_new['chi'])
        row['dx_width_V'] = float(annotation.dx_widths['V'])
        row['dx_width_gamma'] = float(annotation.dx_widths['gamma'])
        row['dx_width_chi'] = float(annotation.dx_widths['chi'])
        row['heating_qdot_max_hi'] = float(annotation.heating_qdot_max_interval.hi) if annotation.heating_qdot_max_interval else np.nan
        row['heating_Q_max_hi'] = float(annotation.heating_Q_max_interval.hi) if annotation.heating_Q_max_interval else np.nan
        row['safety_status'] = str(annotation.safety_status)
    else:
        for key in ['interval_alt_lo','interval_alt_hi','interval_rho_lo','interval_rho_hi','interval_q_lo','interval_q_hi',
                    'width_r_m','width_V_mps','width_gamma_rad','width_chi_rad',
                    'dx_width_V','dx_width_gamma','dx_width_chi',
                    'heating_qdot_max_hi','heating_Q_max_hi']:
            row[key] = np.nan
        for base in ['r', 'phi', 'lam', 'V', 'gamma', 'chi']:
            row[f'{base}_lo'] = np.nan; row[f'{base}_hi'] = np.nan; row[f'{base}_width'] = np.nan
        row['safety_status'] = 'interval_inactive'

    rows.append(row)

    # Track guidance cycles that selected an infeasible heat candidate
    if bool(ctrl_out.guidance_updated):
        sel_heat_feasible = bool(g_debug.get('selected_candidate_heat_feasible', True))
        if not sel_heat_feasible:
            failed_heat_rows.append({
                'step': k, 't_s': t_s,
                'V_mps': x_nom[3], 'alt_m': atm['alt_m'],
                'chosen_sigma_deg': float(g_debug.get('chosen_sigma_cmd_deg', np.nan)),
                'failure_reason': str(g_debug.get('selected_failure_reason', '')),
            })

    x_nom = list(x_nom_next)
    att_nom = step_result.att_new

    if nominal_heat_violation:
        sim_terminated_by_nominal_heat = True
        termination_reason = nominal_heat_violation_reason
        print(f'Stopping at t = {t_s+dt_s:.2f} s: nominal heat violation ({nominal_heat_violation_reason})')
        break

print(f'Loop done. Steps simulated: {len(rows)}.  termination_reason = {termination_reason!r}')

## 4. Save raw data into `revision_v1/`

In [ ]:
df = pd.DataFrame(rows)
failed_heat_df = pd.DataFrame(failed_heat_rows)
thruster_fires_df = pd.DataFrame(thruster_fire_events)
print(f'Trajectory rows: {len(df)}   columns: {len(df.columns)}')
print(f'Failed heat rows: {len(failed_heat_df)}')
print(f'Thruster fire events: {len(thruster_fires_df)}')
df.head()

In [ ]:
# Persist everything
traj_csv = OUTPUT_DIR / 'trajectory.csv'
fires_csv = OUTPUT_DIR / 'thruster_fires.csv'
heat_fail_csv = OUTPUT_DIR / 'failed_heat_cycles.csv'
summary_json = OUTPUT_DIR / 'run_summary.json'

df.to_csv(traj_csv, index=False)
thruster_fires_df.to_csv(fires_csv, index=False)
failed_heat_df.to_csv(heat_fail_csv, index=False)

summary = {
    'trajectory_id': trajectory_id,
    'num_logged_steps': int(len(df)),
    'termination_reason': termination_reason,
    'sim_terminated_by_nominal_heat': bool(sim_terminated_by_nominal_heat),
    'interval_failed_detected': bool(df['interval_failed'].max() > 0) if len(df) else False,
    'interval_recenter_count': int(df['interval_step_recentered'].sum()) if len(df) else 0,
    'interval_split_count': int(df['interval_step_split'].sum()) if len(df) else 0,
    'guidance_updates': int(df['guidance_updated'].sum()) if len(df) else 0,
    'thruster_fire_events': int(len(thruster_fires_df)),
    'final_t_s': float(df['t_s'].iloc[-1]) if len(df) else 0.0,
    'final_alt_m': float(df['alt_m'].iloc[-1]) if len(df) else 0.0,
    'final_V_mps': float(df['V_mps'].iloc[-1]) if len(df) else 0.0,
    'final_gamma_deg': float(math.degrees(df['gamma_rad'].iloc[-1])) if len(df) else 0.0,
    'aero_model': params['aero_model'],
    'target_phi_deg': math.degrees(target_phi_rad),
    'target_lam_deg': math.degrees(target_lam_rad),
    'output_files': {
        'trajectory_csv': str(traj_csv),
        'thruster_fires_csv': str(fires_csv),
        'failed_heat_csv': str(heat_fail_csv),
    },
}
with open(summary_json, 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:')
print(f'  {traj_csv}')
print(f'  {fires_csv}')
print(f'  {heat_fail_csv}')
print(f'  {summary_json}')
summary

# --- CPAS outputs ---
cpas_events_df = pd.DataFrame(cpas_events_log) if cpas_events_log else pd.DataFrame(
    columns=['t_s', 'step', 'alt_m', 'V_mps', 'mach', 'phase', 'event']
)
cpas_events_path = OUTPUT_DIR / 'cpas_events.csv'
cpas_events_df.to_csv(cpas_events_path, index=False)
print(f'  {cpas_events_path}')

cpas_summary = cpas_inst.summary()
summary['cpas_summary'] = cpas_summary
summary['cpas_events_csv'] = str(cpas_events_path)
summary['cpas_total_fires'] = 0  # placeholder, no chute "fires" beyond deploys
summary['cpas_drogue_deploy_t_s'] = cpas_summary['drogue_deploy_t_s']
summary['cpas_pilot_deploy_t_s'] = cpas_summary['pilot_deploy_t_s']
summary['cpas_main_deploy_t_s'] = cpas_summary['main_deploy_t_s']
summary['cpas_landed_t_s'] = cpas_summary['landed_t_s']

# Re-save run_summary.json with the CPAS fields included
with open(OUTPUT_DIR / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'  {OUTPUT_DIR / "run_summary.json"} (with CPAS summary)')


## 5. Trajectory diagnostics

In [ ]:
plotting.plot_ground_track(df, save_fig, target_phi_rad=target_phi_rad, target_lam_rad=target_lam_rad)


In [ ]:
plotting.plot_3d_descent_tube(df, save_fig)


## 6. State time histories with interval uncertainty bands

These reproduce the 'conditional uncertainty propagation' style plots from the original notebook.

In [ ]:
# Helper: shade the interval band where it is finite
def plot_with_band(ax, t, nominal, lo, hi, title, ylabel, nom_color='C0', lo_color='C1', hi_color='C2',
                   nom_label='Nominal', lo_label='Interval low', hi_label='Interval high'):
    ax.plot(t, nominal, color=nom_color, lw=2, label=nom_label)
    mask = lo.notna() & hi.notna()
    if mask.any():
        ax.plot(t[mask], lo[mask], color=lo_color, lw=1.2, label=lo_label)
        ax.plot(t[mask], hi[mask], color=hi_color, lw=1.2, label=hi_label)
        ax.fill_between(t[mask], lo[mask], hi[mask], color=nom_color, alpha=0.15)
    ax.set_title(title); ax.set_xlabel('Time s'); ax.set_ylabel(ylabel)
    ax.legend(loc='best')

In [ ]:
plotting.plot_altitude(df, save_fig)


### Parachute (CPAS) deployment visualization
Altitude with phase-colored shading and deploy event markers; and a terminal-descent zoom showing how chute drag area drives the speed collapse.

In [ ]:
plotting.plot_cpas_altitude_phases(df, save_fig)

In [ ]:
plotting.plot_cpas_speed_dragarea(df, save_fig)

In [ ]:
plotting.plot_speed(df, save_fig)


In [ ]:
plotting.plot_gamma(df, save_fig)


In [ ]:
plotting.plot_chi(df, save_fig)


In [ ]:
plotting.plot_lat_lon(df, save_fig)


In [ ]:
if plotting.want('aero', PLOT_CATEGORIES): plotting.plot_aero_coefficients(df, save_fig)

## 7. Interval supervisor diagnostics

In [ ]:
plotting.plot_state_widths(df, save_fig)


In [ ]:
plotting.plot_density_q(df, save_fig)


## 8. Heating envelope

In [ ]:
plotting.plot_heating_envelope(df, save_fig)


In [ ]:
plotting.plot_heat_shield_map(df, save_fig, nominal_heat_shield=nominal_heat_shield)


## 9. Guidance candidate analysis

In [ ]:
plotting.plot_guidance(df, save_fig)


In [ ]:
plotting.plot_candidate_distribution(df, save_fig)


## 10. RCS firing analysis

Shows the full chain: commanded bank -> tracking error -> torque command -> RCS firing decision.
The thruster raster plot makes individual firings visible.

In [ ]:
plotting.plot_bank_error(df, save_fig)


In [ ]:
plotting.plot_roll_rate_accel(df, save_fig)


In [ ]:
plotting.plot_torque(df, save_fig)


In [ ]:
plotting.plot_duty_vs_fired(df, save_fig)


In [ ]:
plotting.plot_thruster_raster(df, save_fig, thruster_fires_df=thruster_fires_df, rcs_system=rcs_system)


In [ ]:
plotting.plot_firing_rate(df, save_fig, dt_s=dt_s)


In [ ]:
plotting.plot_backlog(df, save_fig)


## 11. Final landing summary

In [ ]:
R_earth = float(constants.RADIUS_EARTH)
dphi = float(df['phi_rad'].iloc[-1] - target_phi_rad)
dlam = float(df['lam_rad'].iloc[-1] - target_lam_rad)
north_err = R_earth * dphi
east_err = R_earth * dlam * math.cos(target_phi_rad)
ground_err = math.sqrt(north_err**2 + east_err**2)

sin_dphi2 = math.sin(dphi/2.0); sin_dlam2 = math.sin(dlam/2.0)
a = sin_dphi2**2 + math.cos(float(df['phi_rad'].iloc[-1]))*math.cos(target_phi_rad)*sin_dlam2**2
a = max(0.0, min(1.0, a))
gc_err = R_earth * 2.0 * math.asin(math.sqrt(a))

landing_summary = {
    'termination_reason': termination_reason,
    'final_t_s': float(df['t_s'].iloc[-1]),
    'final_alt_km': float(df['alt_m'].iloc[-1]/1000.0),
    'final_V_mps': float(df['V_mps'].iloc[-1]),
    'final_gamma_deg': float(math.degrees(df['gamma_rad'].iloc[-1])),
    'final_chi_deg': float(math.degrees(df['chi_rad'].iloc[-1])),
    'final_phi_deg': float(math.degrees(df['phi_rad'].iloc[-1])),
    'final_lam_deg': float(math.degrees(df['lam_rad'].iloc[-1])),
    'north_error_km': north_err/1000.0,
    'east_error_km': east_err/1000.0,
    'flat_ground_error_km': ground_err/1000.0,
    'great_circle_error_km': gc_err/1000.0,
    'rcs_total_fires': int(len(thruster_fires_df)),
    'interval_recenters': int(df['interval_step_recentered'].sum()),
    'interval_splits': int(df['interval_step_split'].sum()),
}
with open(OUTPUT_DIR / 'landing_summary.json', 'w') as f:
    json.dump(landing_summary, f, indent=2)
pd.Series(landing_summary).to_frame('value')